# Homework Modul 3 Vector Search

In [1]:
#%pip install -U sentence-transformers

In [2]:
# Q1 Getting the embeddings Model

from sentence_transformers import SentenceTransformer

model_name = "multi-qa-distilbert-cos-v1"
embedding_model = SentenceTransformer(model_name)



/home/tlucas/.pyenv/versions/3.10.14/envs/llm-zoomcamp/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/home/tlucas/.pyenv/versions/3.10.14/envs/llm-zoomcamp/lib/python3.10/site-packages/torch/cuda/__init__.py:118: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
user_question = "I just discovered the course. Can I still join it?"

v = embedding_model.encode(user_question)

v[0]

0.078222655

What's the first value of the resulting vector?

- -0.24
- -0.04
- **0.07 <--**
- 0.27


In [5]:
# Prepare the documents
import requests 

base_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main'
relative_url = '03-vector-search/eval/documents-with-ids.json'
docs_url = f'{base_url}/{relative_url}?raw=1'
docs_response = requests.get(docs_url)
documents = docs_response.json()


In [21]:
filter_documents = [register for register in documents if register['course'] == "machine-learning-zoomcamp"]

In [23]:
len(filter_documents)

375

In [27]:
# Q2 Creating the embeddings

from tqdm.auto import tqdm
import numpy as np

embeddings = []

for doc in tqdm(filter_documents):
    qa_text = f"{doc['question']}{doc['text']}"
    encode = embedding_model.encode(qa_text)
    embeddings.append(encode)
    
X = np.array(embeddings)

  0%|          | 0/375 [00:00<?, ?it/s]

In [29]:
# What's the shape of X? (X.shape). Include the parantheses.
X.shape

(375, 768)

In [31]:
# Q3 Search
scores = X.dot(v)

In [39]:
# Calculate the maximum score
max_score = np.max(scores)
max_score

0.65065724

What's the highest score in the results?

- 65.0
- 6.5
- **0.65 <--**
- 0.065


In [54]:
# Vector Search

class VectorSearchEngine():
    def __init__(self, documents, embeddings):
        self.documents = documents
        self.embeddings = embeddings

    def search(self, v_query, num_results=10):
        scores = self.embeddings.dot(v_query)
        idx = np.argsort(-scores)[:num_results]
        return [self.documents[i] for i in idx]

search_engine = VectorSearchEngine(documents=filter_documents, embeddings=X)
answers = search_engine.search(v, num_results=5)

In [55]:
answers

[{'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'section': 'General course-related questions',
  'question': 'The course has already started. Can I still join it?',
  'course': 'machine-learning-zoomcamp',
  'id': 'ee58a693'},
 {'text': 'Welcome to the course! Go to the course page (http://mlzoomcamp.com/), scroll down and start going through the course materials. Then read everything in the cohort folder for your cohort’s year.\nClick on the links and start watching the videos. Also watch office hours from previous cohorts. Go to DTC youtube channel and click on Playlists and search for {course yyyy}. ML Zoomcamp was first launched in 2021.\nOr you c

In [41]:
# Q4 Hit-rate for our search engine

import pandas as pd

base_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main'
relative_url = '03-vector-search/eval/ground-truth-data.csv'
ground_truth_url = f'{base_url}/{relative_url}?raw=1'

df_ground_truth = pd.read_csv(ground_truth_url)
df_ground_truth = df_ground_truth[df_ground_truth.course == 'machine-learning-zoomcamp']
ground_truth = df_ground_truth.to_dict(orient='records')

In [50]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

In [77]:
def question_vector(q):
    search_engine = VectorSearchEngine(documents=filter_documents, embeddings=X)
    question = q['question']
    v_q = embedding_model.encode(question)

    return search_engine.search(v_q, num_results=5)

In [78]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return hit_rate(relevance_total)


In [79]:
def documents_embedding(documents):
    embeddings = []
    
    for doc in tqdm(documents):
        qa_text = f"{doc['question']}{doc['course']}"
        encode = embedding_model.encode(qa_text)
        embeddings.append(encode)
        
    X = np.array(embeddings)
    return X

In [80]:
evaluate(ground_truth, question_vector)

  0%|          | 0/1830 [00:00<?, ?it/s]

0.9371584699453552

What did you get?

- **0.93 <--**
- 0.73
- 0.53
- 0.33


In [81]:
# Q5 Indexing with Elasticsearch

from elasticsearch import Elasticsearch

es_client = Elasticsearch('http://localhost:9200') 

index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "id": {"type": "keyword"},
            "question_vector": {
                "type": "dense_vector",
                "dims": 768,
                "index": True,
                "similarity": "cosine"
            },
            "text_vector": {
                "type": "dense_vector",
                "dims": 768,
                "index": True,
                "similarity": "cosine"
            },
            "question_text_vector": {
                "type": "dense_vector",
                "dims": 768,
                "index": True,
                "similarity": "cosine"
            },
        }
    }
}

index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [82]:
for doc in tqdm(filter_documents):
    question = doc['question']
    text = doc['text']
    qt = question + ' ' + text

    doc['question_vector'] = embedding_model.encode(question)
    doc['text_vector'] = embedding_model.encode(text)
    doc['question_text_vector'] = embedding_model.encode(qt)

  0%|          | 0/375 [00:00<?, ?it/s]

In [84]:
for doc in tqdm(filter_documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/375 [00:00<?, ?it/s]

In [107]:
user_question = "I just discovered the course. Can I still join it?"
v = embedding_model.encode(user_question)

In [108]:
search_query = {
    "field": "question_text_vector",
    "query_vector": v,
    "k": 5,
    "num_candidates": 10000,
}

es_results = es_client.search(
    index=index_name,
    knn=search_query,
    source=["text", "section", "question", "course", "id"]
)

In [109]:
es_results['hits']['hits'][0]

{'_index': 'course-questions',
 '_id': 'TOozspABpZv6sghKgfKx',
 '_score': 0.8253288,
 '_source': {'question': 'The course has already started. Can I still join it?',
  'course': 'machine-learning-zoomcamp',
  'section': 'General course-related questions',
  'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'id': 'ee58a693'}}

What's the ID of the document with the highest score? 'ee58a693'

In [98]:
result_docs = [hit['_source'] for hit in es_results['hits']['hits']]

In [100]:
result_docs[0]

{'question': 'The course has already started. Can I still join it?',
 'course': 'machine-learning-zoomcamp',
 'section': 'General course-related questions',
 'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
 'id': 'ee58a693'}

In [110]:
def elastic_search_knn(field, vector):
    knn = {
        "field": field,
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
        }

    search_query = {
        "knn": knn,
        "_source": ["text", "section", "question", "course", "id"]
    }

    es_results = es_client.search(
        index=index_name,
        body=search_query
    )
    
    result_docs = []
    
    for hit in es_results['hits']['hits']:
        result_docs.append(hit['_source'])

    return result_docs

def question_vector_knn(q):
    question = q['question']

    v_q = embedding_model.encode(question)

    return elastic_search_knn('question_text_vector', v_q)

In [111]:
evaluate(ground_truth, question_vector_knn)

  0%|          | 0/1830 [00:00<?, ?it/s]

0.9398907103825137

What's hitrate for our dataset for Elastic?

- **0.93 <--**
- 0.73
- 0.53
- 0.33
